# 03 - Araç Tespiti Fine-Tuning

İstanbul trafik verisiyle YOLOv8 fine-tuning deneyleri.

## Deney Tasarımı
- **Model:** YOLOv8s (COCO pretrained)
- **Veri:** İstanbul trafik görüntüleri (800-1000 etiketli)
- **Hiperparametreler:** 150 epoch, AdamW, lr=0.001, patience=30
- **Karşılaştırma:** Pretrained vs Fine-tuned

In [ ]:
# !pip install ultralytics roboflow

from ultralytics import YOLO
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

## 1. Veri Seti Hazırlığı

In [ ]:
# Roboflow'dan veri seti indirme (opsiyonel)
# from roboflow import Roboflow
# rf = Roboflow(api_key='YOUR_KEY')
# project = rf.workspace().project('istanbul-traffic')
# dataset = project.version(1).download('yolov8')

DATASET_YAML = 'configs/dataset_vehicle.yaml'
print(f'Veri seti config: {DATASET_YAML}')

## 2. Fine-Tuning

In [ ]:
# YOLOv8s Fine-Tuning
model = YOLO('yolov8s.pt')

results = model.train(
    data=DATASET_YAML,
    epochs=150,
    imgsz=640,
    batch=16,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    patience=30,
    save=True,
    save_period=10,
    project='results/training',
    name='yolov8s_istanbul',
    exist_ok=True,
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    mosaic=1.0,
    mixup=0.1,
)

## 3. Eğitim Sonuçları

In [ ]:
# Eğitim metriklerini yükle
results_dir = Path('results/training/yolov8s_istanbul')
csv_path = results_dir / 'results.csv'

if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Loss eğrileri
    axes[0,0].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss')
    axes[0,0].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss')
    axes[0,0].set_title('Box Loss')
    axes[0,0].legend()
    
    axes[0,1].plot(df['epoch'], df['train/cls_loss'], label='Train Cls Loss')
    axes[0,1].plot(df['epoch'], df['val/cls_loss'], label='Val Cls Loss')
    axes[0,1].set_title('Classification Loss')
    axes[0,1].legend()
    
    # mAP
    axes[1,0].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@50')
    axes[1,0].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@50-95')
    axes[1,0].set_title('mAP')
    axes[1,0].legend()
    
    # Precision & Recall
    axes[1,1].plot(df['epoch'], df['metrics/precision(B)'], label='Precision')
    axes[1,1].plot(df['epoch'], df['metrics/recall(B)'], label='Recall')
    axes[1,1].set_title('Precision & Recall')
    axes[1,1].legend()
    
    plt.tight_layout()
    plt.savefig('results/finetuning_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Eğitim sonuçları bulunamadı. Önce eğitimi çalıştırın.')

## 4. Pretrained vs Fine-tuned Karşılaştırması

In [ ]:
# Validation seti üzerinde karşılaştırma
pretrained = YOLO('yolov8s.pt')
finetuned_path = results_dir / 'weights' / 'best.pt'

if finetuned_path.exists():
    finetuned = YOLO(str(finetuned_path))
    
    print('=== Pretrained ===')
    metrics_pre = pretrained.val(data=DATASET_YAML)
    
    print('\n=== Fine-tuned ===')
    metrics_ft = finetuned.val(data=DATASET_YAML)
    
    # Karşılaştırma tablosu
    comparison = pd.DataFrame({
        'Metric': ['mAP@50', 'mAP@50-95', 'Precision', 'Recall'],
        'Pretrained': [
            metrics_pre.box.map50, metrics_pre.box.map,
            metrics_pre.box.mp, metrics_pre.box.mr
        ],
        'Fine-tuned': [
            metrics_ft.box.map50, metrics_ft.box.map,
            metrics_ft.box.mp, metrics_ft.box.mr
        ],
    })
    comparison['Fark'] = comparison['Fine-tuned'] - comparison['Pretrained']
    print('\n', comparison.to_string(index=False))
    comparison.to_csv('results/pretrained_vs_finetuned.csv', index=False)
else:
    print('Fine-tuned model bulunamadı.')